# Machine Learning Modeling

In [1]:
# Make sure you are at the parent directory
from pathlib import Path
import sys

# Define MODE
MODE = "EXPANSE" # Either "COLAB", "LOCAL", "EXPANSE"
if MODE.upper() not in ('EXPANSE', 'COLAB', 'LOCAL'):
    raise Exception("Invalid mode, the only acceptible are 'EXPANSE', 'COLAB', 'LOCAL'")
if MODE.upper() == "COLAB":
    from google.colab import drive
    drive.mount('/content/drive')

# Root path by MODE
PROJECT_ROOT_BY_MODE = {
    "COLAB": Path("/content/drive/MyDrive/DSC 288R/Project"),
    "EXPANSE": Path("/home/bguo3/bguo3/DSC-288R-Capstone-Final-Project"),
    "LOCAL": Path("/Users/steveg/Desktop/DSC-288R-Capstone-Final-Project"),
}
ROOT = PROJECT_ROOT_BY_MODE[MODE.upper()]

# Add the root path to global system
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT.resolve()))

# Print the root path
print("MODE:", MODE)
print("PROJECT_ROOT:", ROOT)

MODE: EXPANSE
PROJECT_ROOT: /home/bguo3/bguo3/DSC-288R-Capstone-Final-Project


## Import Modules

In [2]:
from src.utils.pyspark_utils import create_spark_session, memory_count
from src.utils.paths_utils import ProjectPaths
from src.utils.io_utils import read_spark_parquet, write_spark_split_parquets
import src.pipelines.ML_modeling as ml

import time

Matplotlib created a temporary cache directory at /scratch/bguo3/job_49075104/matplotlib-i0bj1n4h because the default path (/home/jovyan/.cache/matplotlib) is not a writable directory; it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


## Read Train/Validation/Test Full/Sampled Dataset From Parquet

In [3]:
# Set up for Spark app & resource allocation
spark = create_spark_session("steam_reviews_machine_learning_modeling")

# Load data
paths = ProjectPaths(MODE)
train = read_spark_parquet(spark=spark, path=paths.random_row_train_parquet)
val = read_spark_parquet(spark=spark, path=paths.random_row_val_parquet)
test = read_spark_parquet(spark=spark, path=paths.random_row_test_parquet)

Read Spark parquet from: /expanse/lustre/scratch/bguo3/temp_project/steam_reviews/splits/random_row/train_parquet
Read Spark parquet from: /expanse/lustre/scratch/bguo3/temp_project/steam_reviews/splits/random_row/val_parquet
Read Spark parquet from: /expanse/lustre/scratch/bguo3/temp_project/steam_reviews/splits/random_row/test_parquet


In [4]:
# Quick Feat Engineering here
def feat_label(df):
    from pyspark.ml.feature import VectorAssembler
    from pyspark.sql import functions as F
    
    feature_cols = [
        "author_num_games_owned",
        "author_num_reviews",
        "author_playtime_forever",
        "author_playtime_last_two_weeks",
        "author_playtime_at_review",
        "author_last_played",
        "voted_up",
        "votes_up",
        "votes_funny",
        "weighted_vote_score",
        "comment_count",
        "written_during_early_access",
        "timestamp_created",
        "timestamp_updated",
    ]
    
    feat_df = (
        df
        .withColumn('churn', 
            F.when(F.col('author_playtime_last_two_weeks') > 60, 1)
            .otherwise(0)
        )
        .select(feature_cols + ['churn'])
        
    )

    final_df = (
        VectorAssembler(inputCols=feature_cols, outputCol='finalized_features')
        .transform(feat_df)
        .select('finalized_features', 'churn')
    )
        
    final_df.printSchema()
    memory_count(final_df, include_row_count=True)

    return final_df

train = train.transform(feat_label)
val = val.transform(feat_label)

root
 |-- finalized_features: vector (nullable = true)
 |-- churn: integer (nullable = false)

Total row counts: 76405108 rows
Total estimated size: 2.57 GB
root
 |-- finalized_features: vector (nullable = true)
 |-- churn: integer (nullable = false)

Total row counts: 16373962 rows
Total estimated size: 0.55 GB


## Logistic Regression Model

In [5]:
start = time.time()

lr, lr_train_pred, lr_val_pred = ml.fit_transform_model(
    model_name ='log_reg',
    train_final_df = train,
    test_final_df = val,
    spark = spark,
    cache = False
)
ml.evaluate_model(
    model_name= "log_reg",
    train_pred_df= lr_train_pred,
    test_pred_df= lr_val_pred,
    verbose= True
)
    
elapsed = time.time() - start
print(f"ML Logistic Regression took: {elapsed:.2f} seconds")

log_reg Train areaUnderROC: 1.0000
log_reg Test areaUnderROC: 1.0000

log_reg Train areaUnderPR: 1.0000
log_reg Test areaUnderPR: 1.0000

log_reg Train f1: 0.9997
log_reg Test f1: 0.9997

log_reg Train weightedPrecision: 0.9997
log_reg Test weightedPrecision: 0.9997

log_reg Train weightedRecall: 0.9997
log_reg Test weightedRecall: 0.9997

log_reg Train accuracy: 0.9997
log_reg Test accuracy: 0.9997

ML Logistic Regression took: 742.78 seconds


## Decision Tree Model

In [ ]:
start = time.time()

dt, dt_train_pred, dt_val_pred = ml.fit_transform_model(
    model_name ='decision_tree',
    train_final_df = train,
    test_final_df = val,
    spark = spark,
    cache = False
)
ml.evaluate_model(
    model_name= "decision_tree",
    train_pred_df= lr_train_pred,
    test_pred_df= lr_val_pred,
    verbose= True
)
    
elapsed = time.time() - start
print(f"ML Decision Tree took: {elapsed:.2f} seconds")

## Random Forest Model

In [ ]:
start = time.time()

rf, rf_train_pred, rf_val_pred = ml.fit_transform_model(
    model_name ='decision_tree',
    train_final_df = train,
    test_final_df = val,
    spark = spark,
    cache = False
)
ml.evaluate_model(
    model_name= "decision_tree",
    train_pred_df= rf_train_pred,
    test_pred_df= rf_val_pred,
    verbose= True
)
    
elapsed = time.time() - start
print(f"ML Random Forest took: {elapsed:.2f} seconds")

In [ ]:
# # Temp preprocess data to avoid XGBoost error for N class expecting class value starting at 0
# xgb_train_final = train_final.withColumn("label", F.col("label").cast("int"))
# xgb_val_final   = val_final.withColumn("label", F.col("label").cast("int"))

# xgb, xgb_train_pred, xgb_val_pred = helper_fit_transform(
#     'xgb', 
#     xgb_train_final, xgb_val_final
# )
# helper_evaluate_model(
#     'xgb', 
#     xgb_train_pred, xgb_val_pred
# )

# elapsed = time.time() - start
# print(f"ML XGBoost took: {elapsed:.2f} seconds")


In [ ]:
spark.stop()